## 05 - Monitoring and Model Registry (Skeleton)

This notebook is a placeholder for setting up:

1. Model registration with SageMaker Model Registry  
2. Model and data monitoring (drift detection, baseline checks)


## Compress model for Sagemaker

In [1]:
import os
import shutil
import tarfile

# Ensure directories exist
os.makedirs("model", exist_ok=True)
os.makedirs("registry", exist_ok=True)

# Copy lr_model.pkl to model.pkl
src_path = "model/lr_model.pkl"
dst_path = "model/model.pkl"
shutil.copyfile(src_path, dst_path)
print("Copied lr_model.pkl to model.pkl")

# Compress the pickle model into tar.gz format
with tarfile.open("registry/model.tar.gz", "w:gz") as tar:
    tar.add("model/model.pkl", arcname="model.pkl")
    tar.add("inference.py", arcname="inference.py")

print("Compressed model into registry/model.tar.gz")


Copied lr_model.pkl to model.pkl
Compressed model into registry/model.tar.gz


## Sagemaker Setup

In [2]:
from sagemaker import Session

session = Session()
bucket = session.default_bucket()
model_key = "diabetes/registry/model.tar.gz"

# Upload tar.gz to S3
model_s3_uri = session.upload_data(
    path="registry/model.tar.gz",
    bucket=bucket,
    key_prefix="diabetes/registry"
)

print("Uploaded model to:", model_s3_uri)


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Uploaded model to: s3://sagemaker-us-east-1-380537322556/diabetes/registry/model.tar.gz


## Create Model Package Group

In [3]:
import boto3
import sagemaker
from sagemaker import image_uris

model_package_group_name = "ReadmissionModelGroup"

# Get the Scikit-learn container URI for current region
region = session.boto_region_name
sklearn_uri = image_uris.retrieve(framework="sklearn", region=region, version="1.0-1")

sm_client = boto3.client("sagemaker")

# Check if model package group already exists
existing_groups = sm_client.list_model_package_groups(
    NameContains=model_package_group_name
).get("ModelPackageGroupSummaryList", [])

if any(group["ModelPackageGroupName"] == model_package_group_name for group in existing_groups):
    print("Model package group already exists:", model_package_group_name)
else:
    sm_client.create_model_package_group(
        ModelPackageGroupName=model_package_group_name,
        ModelPackageGroupDescription="Logistic Regression for hospital readmission prediction"
    )
    print("Created model package group:", model_package_group_name)


Model package group already exists: ReadmissionModelGroup


## Register Model

In [4]:
# Register the model version
response = sm_client.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription="Logistic Regression model (C=0.01, class_weight=balanced)",
    InferenceSpecification={
        "Containers": [{
            "Image": sklearn_uri,
            "ModelDataUrl": model_s3_uri
        }],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"]
    },
    ModelApprovalStatus="PendingManualApproval"
)

print("Registered model version in:", model_package_group_name)
print("Model Package ARN:", response["ModelPackageArn"])


Registered model version in: ReadmissionModelGroup
Model Package ARN: arn:aws:sagemaker:us-east-1:380537322556:model-package/ReadmissionModelGroup/9


## Model Monitoring

This section defines the expected structure for setting up model monitoring using Amazon SageMaker and CloudWatch.

The monitoring pipeline should include:

1. **Endpoint deployment with data capture enabled**
2. **Baseline generation using validation data** (produces statistics + constraints)
3. **Scheduling a model quality monitoring job**
4. **Capturing predictions and ground truth**
5. **Alerting via CloudWatch**
6. **Monitoring bias, explainability, and infrastructure metrics**


### Monitoring S3 Paths

The following folders should be used for monitoring-related artifacts:

- `s3://{bucket}/diabetes/monitoring/capture/` – Captured inference data
- `s3://{bucket}/diabetes/monitoring/baseline/` – Baseline statistics and constraints
- `s3://{bucket}/diabetes/monitoring/groundtruth/` – Human-labeled or synthetic ground truth labels
- `s3://{bucket}/diabetes/monitoring/reports/` – Output reports from monitoring jobs


In [ ]:
# TODO: Deploy model endpoint with data capture enabled

# from sagemaker.model import Model
# from sagemaker.model_monitor import DataCaptureConfig

# capture_config = DataCaptureConfig(
#     enable_capture=True,
#     sampling_percentage=100,
#     destination_s3_uri=f"s3://{bucket}/diabetes/monitoring/capture"
# )

# model = Model(...)
# predictor = model.deploy(initial_instance_count=1, instance_type="ml.m5.large", data_capture_config=capture_config)


In [ ]:
# TODO: Run baseline processing job using validation data

# from sagemaker.model_monitor import DefaultModelMonitor

# monitor = DefaultModelMonitor(role=role, instance_count=1, instance_type="ml.m5.large")

# monitor.suggest_baseline(
#     baseline_dataset=f"s3://{bucket}/diabetes/monitoring/baseline/validation_with_labels.csv",
#     dataset_format=DatasetFormat.csv(header=True),
#     output_s3_uri=f"s3://{bucket}/diabetes/monitoring/baseline/",
#     wait=True
# )


In [ ]:
# TODO: Schedule model quality monitoring job and optionally configure CloudWatch alerts

# monitor.create_monitoring_schedule(
#     monitor_schedule_name="readmission-monitor",
#     endpoint_input=predictor.endpoint_name,
#     output_s3_uri=f"s3://{bucket}/diabetes/monitoring/reports/"
# )

# Use CloudWatch to trigger alarms on accuracy/F1/AUC drift, latency, and infra usage


In [31]:
# === IMPORTS ===
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.properties import PropertyFile
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.sklearn.processing import SKLearnProcessor

# === SKLearn Processor for Evaluation ===
sklearn_processor = SKLearnProcessor(
    framework_version="0.23-1",  # or "1.0-1" if needed
    role=sagemaker.get_execution_role(),
    instance_type="ml.m5.large",
    instance_count=1
)

# === Evaluation Step ===
evaluation_report = PropertyFile(
    name="evaluation",
    output_name="output",
    path="evaluation.json"
)

step_eval = ProcessingStep(
    name="ModelEvaluation",
    processor=sklearn_processor,
    code="/home/sagemaker-user/AAI540_FinalProject/04_evaluation_and_reporting.py",
    property_files=[evaluation_report]
)

# === Model Object from Training Output ===
model = Model(
    image_uri=estimator.image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=sagemaker.get_execution_role()
)

# === Attach Metrics for Model Registry ===
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=step_eval.properties.ProcessingOutputConfig.Outputs["output"].S3Output.S3Uri,
        content_type="application/json"
    )
)

# === Register the Model (Always Registers) ===
step_register = RegisterModel(
    name="RegisterModel",
    estimator=estimator,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name="YourModelPackageGroupName",
    model_metrics=model_metrics
)


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
